# Notebook 3b: Improved ANN Model
## Optimized Architecture & Hyperparameters

**Improvements:**
- Deeper network architecture
- Batch normalization
- Learning rate scheduler
- Early stopping
- More epochs
- Better regularization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Data

In [ ]:
X_train = pd.read_csv('../data/X_train_pca.csv')
X_test = pd.read_csv('../data/X_test_pca.csv')
y_train = pd.read_csv('../data/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/y_test.csv').values.ravel()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 2. Create Tensors and DataLoaders

In [ ]:
X_train_tensor = torch.FloatTensor(X_train.values).to(device)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1).to(device)
X_test_tensor = torch.FloatTensor(X_test.values).to(device)
y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## 3. Improved ANN Architecture

In [ ]:
class ImprovedCornANN(nn.Module):
    def __init__(self, input_size):
        super(ImprovedCornANN, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.fc4 = nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)
        self.fc5 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)
        x = self.relu(self.bn4(self.fc4(x)))
        x = self.fc5(x)
        return x

model = ImprovedCornANN(X_train.shape[1]).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

## 4. Training Setup with Scheduler

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

print("Optimizer: Adam with weight decay")
print("Scheduler: ReduceLROnPlateau")
print("Initial LR: 0.001")

## 5. Training with Early Stopping

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
epochs = 300
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 30
patience_counter = 0

print("Training with early stopping...\n")
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = validate_epoch(model, test_loader, criterion)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), '../models/best_improved_ann.pth')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Train: {train_loss:.4f}, Val: {val_loss:.4f}, Best: {best_val_loss:.4f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(torch.load('../models/best_improved_ann.pth'))
print("\n✓ Training completed! Best model loaded.")

## 6. Visualize Training

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Improved ANN Training Progress')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/improved_ann_training.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    y_train_pred = model(X_train_tensor).cpu().numpy()
    y_test_pred = model(X_test_tensor).cpu().numpy()

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*60)
print("IMPROVED ANN PERFORMANCE")
print("="*60)
print("\nTraining Set:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  R²:   {train_r2:.4f}")
print("\nTest Set:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  R²:   {test_r2:.4f}")
print("="*60)

## 8. Compare with Original ANN

In [ ]:
original_ann = pd.read_csv('../outputs/ann_results.csv')

comparison = pd.DataFrame({
    'Model': ['Original ANN', 'Improved ANN'],
    'Test_R2': [original_ann['Test_R2'].values[0], test_r2],
    'Test_RMSE': [original_ann['Test_RMSE'].values[0], test_rmse],
    'Test_MAE': [original_ann['Test_MAE'].values[0], test_mae]
})

print("\nComparison:")
print(comparison.to_string(index=False))

improvement_r2 = ((test_r2 - original_ann['Test_R2'].values[0]) / original_ann['Test_R2'].values[0]) * 100
print(f"\nR² Improvement: {improvement_r2:+.2f}%")

## 9. Predictions Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_train, y_train_pred, alpha=0.6, edgecolors='black', s=50)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Training Set (R² = {train_r2:.4f})')
axes[0].grid(alpha=0.3)

axes[1].scatter(y_test, y_test_pred, alpha=0.6, edgecolors='black', s=50, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Test Set (R² = {test_r2:.4f})')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/improved_ann_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Save Results

In [ ]:
results = pd.DataFrame({
    'Model': ['Improved ANN'],
    'Train_RMSE': [train_rmse],
    'Train_MAE': [train_mae],
    'Train_R2': [train_r2],
    'Test_RMSE': [test_rmse],
    'Test_MAE': [test_mae],
    'Test_R2': [test_r2]
})

results.to_csv('../outputs/improved_ann_results.csv', index=False)
print("✓ Results saved!")
print("\n" + "="*60)
print(f"Final Test R²: {test_r2:.4f}")
print("="*60)